In [0]:
%sql
-- 1
CREATE OR REPLACE TABLE cyntexa_dev.sales.finance_demo (
    transaction_id INT,
    customer_id INT,
    transaction_date DATE,
    amount DOUBLE,
    payment_method STRING,
    status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.finance_demo
VALUES
(1, 101, '2026-08-01', 5000.00, 'UPI', 'COMPLETED'),
(2, 102, '2026-08-01', 7500.00, 'CARD', 'COMPLETED'),
(3, 103, '2026-08-01', 700.00, 'CARD', 'COMPLETED');

In [0]:
%sql
UPDATE cyntexa_dev.sales.finance_demo
SET amount = 5500.00
WHERE transaction_id = 1;

In [0]:
%sql

INSERT INTO cyntexa_dev.sales.finance_demo
VALUES
(4, 105, '2026-08-02', 3200.00, 'CASH', 'PENDING');

In [0]:
%sql
DESCRIBE HISTORY cyntexa_dev.sales.finance_demo


In [0]:
%sql
-- 2
COPY INTO cyntexa_dev.sales.finance_demo
FROM '/Volumes/cyntexa_dev/sales/raw/finance/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'true'
);

In [0]:
%sql
SELECT * FROM cyntexa_dev.sales.finance_demo

In [0]:
%sql
-- 3
SELECT * FROM cyntexa_dev.sales.finance_demo VERSION AS OF 3

In [0]:
%sql
SELECT * FROM cyntexa_dev.sales.finance_demo TIMESTAMP AS OF '2026-08-26T11:57:07.000+00:00'

#INTERMEDIATE

In [0]:
# 4.

from pyspark.sql.functions import col

df = spark.createDataFrame([
    (4, 104, "2026-08-03", 4500.00, "UPI", "COMPLETED", "INR")
], [
    "transaction_id",
    "customer_id",
    "transaction_date",
    "amount",
    "payment_method",
    "status",
    "currency"
])

df = df.withColumn("transaction_date", col("transaction_date").cast("date"))

df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("cyntexa_dev.sales.finance_demo")

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("transaction_id", col("transaction_id").cast("int")) \
      .withColumn("customer_id", col("customer_id").cast("int"))

df.write \
    .format("delta") \
    .mode("append") \
    .option("overwriteSchema","true") \
    .saveAsTable("cyntexa_dev.sales.finance_demo")

### mergeSchema
- It merges the new schema with the existing table schema.
- When a new column is added, existing records get NULL for that column because they don't have a value for it.
- Commonly used when adding new columns.

### overwriteSchema
- It replaces the existing table schema with the schema of the DataFrame being written.
- It is used when we need to change an existing column's datatype or make other schema changes.

In [0]:
#5.

from pyspark.sql.functions import current_timestamp

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/cyntexa_dev/sales/raw/finance/_schema") \
    .load("/Volumes/cyntexa_dev/sales/raw/finance/")

df = df.withColumn("_ingested_at", current_timestamp())

query = df.writeStream \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "/Volumes/cyntexa_dev/sales/raw/finance/_checkpoint") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .toTable("cyntexa_dev.sales.finance_demo")
query.recentProgress

In [0]:
%sql
-- 6.
RESTORE cyntexa_dev.sales.finance_demo TO VERSION AS OF 5

-- the versions i created after it will remain in storage as it is. they still present as history.
-- we can use time travel to read them, subject to Delta s retention/VACUUM rules.


#ADVANCED

7. 
### CTAS 
- Used mainly to create a table from an existing result.
- It is not an incremental file-ingestion mechanism.
- Cost: Low to medium, depending on the amount of data processed.
- Latency: Usually higherbecause data is processed when the CTAS query runs.
- Operational complexity: Low.

### COPY INTO
- Used for incremental file ingestion.
- It keeps track of files that have already been processed and avoids processing them again.
- When new files arrive, it can load only those new files.
- Cost: low to medium.
- Latency: Medium; it depends on how frequently it is executed.
- Operational complexity: Low to medium.

### Auto Loader
- Used for incremental and continuous file ingestion.
- It detects new files as they arrive and processes them incrementally.
- Cost:  higher because the pipeline is continuously running.
- Latency: Low.
- Operational complexity: Medium.

### Lakeflow Declarative Pipelines
- Used to build managed data pipelines for batch and streaming workloads.
- It is useful when the business requires continuously updated data or real time processing.
- Cost: Depends on the pipeline configuration and workload.
- Latency: very low 
- Operational complexity: Lower than managing a complex streaming pipeline because Databricks manages much of the pipeline infrastructure.

###cyntexa -

Cyntexa should move with the Autoloader because ingestion data as file source arrives unpredictably throughout the day so it automatically process that.no manual run needed.


8. Design a recovery runbook: if a bad file corrupts the silver table at 2am, walk through the exact
commands (DESCRIBE HISTORY, RESTORE or time travel + overwrite) an on-call engineer would run.
9. (Data Analyst) Using DESCRIBE HISTORY, produce a 'data freshness' report showing how frequently a
given table is actually updated, to validate an SLA claim made to a business stakeholder.8. 

In [0]:
%sql
DESCRIBE HISTORY cyntexa_dev.sales.sales_silver;

In [0]:
%sql
-- 8.

DESCRIBE HISTORY cyntexa_dev.sales.finance_demo;
SELECT * FROM cyntexa_dev.sales.finance_demo VERSION AS OF 4;
RESTORE TABLE cyntexa_dev.sales.finance_demo TO VERSION OF 4;


In [0]:
%sql
describe history cyntexa_dev.sales.finance_demo

In [0]:
%sql
select
  version,
  timestamp,
  operation,
  lag(timestamp) over (order by timestamp) as previous_update,
  timestamp - lag(timestamp) over (order by timestamp) as time_since_last_update
from (
  describe history cyntexa_dev.sales.finance_demo
)
order by timestamp;